# 06: Joint Final Evaluation & Comparative Benchmark
Comprehensive model comparison across all 6 architectures evaluated on the held-out test set ($N=338$).

1. **Baseline Custom CNN** (from scratch, 4-block CNN)
2. **ResNet-18** (Residual Connections)
3. **MobileNetV3-Small** (Squeeze-and-Excitation Attention)
4. **CRNN** (CNN + Bi-GRU + Temporal Attention, trained from scratch)
5. **EfficientNet-B0** (Inverted Residual MBConv + Compound Scaling)
6. **ConvNeXt-Tiny** (Modernized 7x7 Depthwise Separable CNN)


In [ ]:
# Cell 1: Ingest All Model Evaluation Artifacts
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'code' else Path('.').resolve()
MODELS_DIR = PROJECT_ROOT / 'model_artifacts'

models_data = [
    ('Baseline Custom CNN', MODELS_DIR / 'custom_cnn' / 'test_evaluation_report.json', 'Center Crop', 423727),
    ('ResNet-18', MODELS_DIR / 'optimized_resnet18' / 'test_evaluation_report.json', 'Center Crop', 11184143),
    ('MobileNetV3 (Center)', MODELS_DIR / 'optimized_mobilenet' / 'test_evaluation_report.json', 'Center Crop', 1674863),
    ('MobileNetV3 (Ensemble)', MODELS_DIR / 'optimized_mobilenet' / 'final_evaluation_report.json', 'Multi-Window Ensemble', 1674863),
    ('CRNN (Ensemble)', MODELS_DIR / 'crnn' / 'final_evaluation_report.json', 'Multi-Window Ensemble', 786800),
    ('EfficientNet-B0 (Ensemble)', MODELS_DIR / 'efficientnet_b0' / 'final_evaluation_report.json', 'Multi-Window Ensemble', 4339595),
    ('ConvNeXt-Tiny (Ensemble)', MODELS_DIR / 'convnext_tiny' / 'final_evaluation_report.json', 'Multi-Window Ensemble', 28021359),
    ('BirdNET (Cornell SOTA)', MODELS_DIR / 'birdnet' / 'final_evaluation_report.json', 'Sliding Window', 'Pretrained Foundation'),
]

rows = []
for name, path, method, params in models_data:
    if path.exists():
        with open(path) as f: rep = json.load(f)
        acc = rep.get('test_accuracy_top1', rep.get('test_accuracy', 0)) * 100
        top3 = rep.get('test_accuracy_top3', np.nan)
        top3 = top3 * 100 if not np.isnan(top3) else '-'
        f1 = rep.get('test_macro_f1', 0) * 100
        rows.append({
            'Model Architecture': name,
            'Inference Method': method,
            'Parameters': f'{params:,}',
            'Top-1 Test Acc (%)': round(acc, 2),
            'Top-3 Test Acc (%)': round(top3, 2) if isinstance(top3, float) else top3,
            'Macro F1 (%)': round(f1, 2)
        })

df_comp = pd.DataFrame(rows)
display(df_comp)


In [ ]:
# Cell 2: Comparative Bar Chart Visualization
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df_comp))
width = 0.35

accs = df_comp['Top-1 Test Acc (%)'].values
f1s = df_comp['Macro F1 (%)'].values

rects1 = ax.bar(x - width/2, accs, width, label='Top-1 Accuracy (%)', color='royalblue')
rects2 = ax.bar(x + width/2, f1s, width, label='Macro F1 (%)', color='forestgreen')

ax.set_ylabel('Score (%)')
ax.set_title('Comparative Benchmark on Holdout Test Set (N=338)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(df_comp['Model Architecture'], rotation=25, ha='right')
ax.set_ylim(0, 100)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in rects1: ax.annotate(f'{bar.get_height():.1f}%', (bar.get_x() + bar.get_width()/2, bar.get_height()), ha='center', va='bottom', fontsize=8)
for bar in rects2: ax.annotate(f'{bar.get_height():.1f}%', (bar.get_x() + bar.get_width()/2, bar.get_height()), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'comparative_model_benchmark.png', dpi=200)
plt.show()


In [ ]:
# Cell 3: Interactive Single-Audio Inference Test
!python test_model.py --model efficientnet --mode predict --audio sample_dataset/audio/comkin1/XC110343.ogg
